In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from gravpop import *
import h5py
import numpy as np
import pandas as pd

### For saving and loading
### should be in newest gravpop version
_NONE_ATTR = "__is_none__"

def save_dict_h5(filename, data):
    def _save_group(h5group, d):
        for k, v in d.items():
            if v is None:
                g = h5group.create_group(k)
                g.attrs[_NONE_ATTR] = True
            elif isinstance(v, dict):
                _save_group(h5group.create_group(k), v)
            elif isinstance(v, pd.DataFrame):
                g = h5group.create_group(k)
                g.create_dataset("columns", data=np.array(v.columns, dtype="S"))
                g.create_dataset("values", data=v.to_numpy())
            elif isinstance(v, (list, tuple)):
                arr = np.array(v)
                if arr.dtype.kind in {"U", "O"}:  # strings (or objects that are strings)
                    arr = arr.astype("S")
                h5group.create_dataset(k, data=arr)
            elif isinstance(v, str):
                h5group.create_dataset(k, data=np.bytes_(v))  # NumPy 2.0+
            else:
                h5group.create_dataset(k, data=np.array(v))
    with h5py.File(filename, "w") as f:
        _save_group(f, data)

def load_dict_h5(filename):
    def _load_group(h5group):
        out = {}
        for k, v in h5group.items():
            if isinstance(v, h5py.Group):
                # None sentinel?
                if v.attrs.get(_NONE_ATTR, False):
                    out[k] = None
                # DataFrame?
                elif "columns" in v and "values" in v:
                    cols = [c.decode() for c in v["columns"][()]]
                    out[k] = pd.DataFrame(v["values"][()], columns=cols)
                else:
                    out[k] = _load_group(v)
            else:
                arr = v[()]
                if arr.dtype.kind == "S":
                    # Return string arrays as Python lists of str; scalars as str
                    out[k] = arr.decode() if arr.ndim == 0 else [x.decode() for x in arr.flatten()]
                else:
                    out[k] = arr.tolist() if arr.ndim == 0 else arr
        return out
    with h5py.File(filename, "r") as f:
        return _load_group(f)

/home/noah.wolfe/.conda/envs/just-for-kicks/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Data

In [ ]:
event_data = load_dict_h5("/home/asad.hussain/O4b_test/o4a_data_products/event_data.hdf5");
selection_data = load_dict_h5("/home/asad.hussain/O4b_test/o4a_data_products/selection_data.h5");
analysis_time = selection_data.pop('analysis_time')
total_generated = selection_data.pop('total_generated')
total_detected = selection_data.pop('total_detected')
selection_func = SelectionFunction(selection_data, analysis_time=analysis_time, total_generated=total_generated, total_detected=total_detected);

# Define Models

In [9]:
smoothed_mass_model = SmoothedTwoComponentPrimaryMassRatio(mmin_fixed=2, mmax_fixed=300, gaussian_mass_maximum=100,
                                 var_names=['mass_1_source', 'mass_ratio'],
                                 hyper_var_names=['alpha', 'beta', 'lam', 'mpp', 'sigpp', 'delta_m', 'mmin', 'mmax'])
    

redshift_model = PowerLawRedshift(var_names=['redshift'],hyper_var_names=['lamb'], z_max=3);

the_spin_orientation_model_2D = GaussianIsotropicSpinOrientationsIIDAnalytic2D(
    a=-1, b=1,
    var_names=['cos_tilt_1', 'cos_tilt_2'],
    hyper_var_names=['xi_spin','sigma_spin']
)

the_spin_orientation_model_1D = GaussianIsotropicSpinOrientationsIIDAnalytic(
    a=-1, b=1,
    var_names=['cos_tilt_1', 'cos_tilt_2'],
    hyper_var_names=['xi_spin','sigma_spin']
)


chi_model = TruncatedGaussian1DAnalytic(
    var_names=['chi_1', 'chi_2'],
    hyper_var_names=['mu_chi', 'sigma_chi'],
    a=0,
    b=0,

)

"""
var_names = ['chi_1', 'chi_2'];
a = [0,0]; b=[1,1];

spin_2D_a = TruncatedGaussian2DAnalytic(
    var_names=var_names,
    hyper_var_names=['mu_chi_1_at_0', 'sigma_chi_1_at_0', 'mu_chi_2_at_0', 'sigma_chi_2_at_0', 'rho_chi_1'],
    a=a,
    b=b
)
spin_2D_b = TruncatedGaussian2DAnalytic(
    var_names=var_names,
    hyper_var_names=['mu_chi_1', 'sigma_chi_1', 'mu_chi_2', 'sigma_chi_2', 'rho_chi_2'],
    a=a,
    b=b
)

full_spin_model = mixture(
    [spin_2D_a, spin_2D_b],
    ['eta_spin', 'one_minus_eta_spin']
)
"""

"\nvar_names = ['chi_1', 'chi_2'];\na = [0,0]; b=[1,1];\n\nspin_2D_a = TruncatedGaussian2DAnalytic(\n    var_names=var_names,\n    hyper_var_names=['mu_chi_1_at_0', 'sigma_chi_1_at_0', 'mu_chi_2_at_0', 'sigma_chi_2_at_0', 'rho_chi_1'],\n    a=a,\n    b=b\n)\nspin_2D_b = TruncatedGaussian2DAnalytic(\n    var_names=var_names,\n    hyper_var_names=['mu_chi_1', 'sigma_chi_1', 'mu_chi_2', 'sigma_chi_2', 'rho_chi_2'],\n    a=a,\n    b=b\n)\n\nfull_spin_model = mixture(\n    [spin_2D_a, spin_2D_b],\n    ['eta_spin', 'one_minus_eta_spin']\n)\n"

# Define Priors

In [10]:
mass_redshift_priors = dict(
    alpha 			= dist.Uniform(-4,12),
    lam 			= dist.Uniform(0,1),
    mmin 			= dist.Uniform(2,6),
    mmax 			= DiracDelta(300),
    beta 			= dist.Uniform(-2,7),
    mpp 			= dist.Uniform(20,50),
    sigpp 			= dist.Uniform(1,10),
    delta_m 		= dist.Uniform(0,12),
    lamb 			= dist.Uniform(-10,10)
)

spin_mag_priors = dict(
    mu_chi = dist.Uniform(0, 1),
    sigma_chi = dist.Uniform(0, 1),
)

spin_orientation_standard_priors = dict(
    xi_spin 	= dist.Uniform(0,1),
    sigma_spin 	= dist.Uniform(0.2,4)
)

priors = mass_redshift_priors.copy()
priors.update(spin_mag_priors)
priors.update(spin_orientation_standard_priors)

In [ ]:
priors['xi_spin']

{'alpha': <numpyro.distributions.continuous.Uniform object at 0x7f91306b3210 with batch shape () and event shape ()>,
 'lam': <numpyro.distributions.continuous.Uniform object at 0x7f95af303f10 with batch shape () and event shape ()>,
 'mmin': <numpyro.distributions.continuous.Uniform object at 0x7f907e3bb2d0 with batch shape () and event shape ()>,
 'mmax': <gravpop.sampler.sampler.DiracDelta at 0x7f907e3bb1d0>,
 'beta': <numpyro.distributions.continuous.Uniform object at 0x7f907e3ba3d0 with batch shape () and event shape ()>,
 'mpp': <numpyro.distributions.continuous.Uniform object at 0x7f907e3ba090 with batch shape () and event shape ()>,
 'sigpp': <numpyro.distributions.continuous.Uniform object at 0x7f907e3ba810 with batch shape () and event shape ()>,
 'delta_m': <numpyro.distributions.continuous.Uniform object at 0x7f907e3b9e50 with batch shape () and event shape ()>,
 'lamb': <numpyro.distributions.continuous.Uniform object at 0x7f907e3ba350 with batch shape () and event shape (

# Define Likelihood

In [11]:
HL = MarginalizedHybridLikelihood(
    event_data=event_data,
    selection_data=selection_func,
    models=[smoothed_mass_model, redshift_model, the_spin_orientation_model_2D, chi_model],
    models_selection=[smoothed_mass_model, redshift_model, the_spin_orientation_model_1D, chi_model],
    fix_kernels_selection={},
    fix_kernels_events={}
)

In [17]:
selection_func?

Type:        SelectionFunction
String form: SelectionFunction(selection_data={'chi_1': array([0.52948237, 0.17152437, 0.03466602, ..., 0.6439 <...> time=2.123025832129186, total_generated=1130738485.0, redshift_model=None, total_detected=983807)
File:        ~/.conda/envs/just-for-kicks/lib/python3.11/site-packages/gravpop/hyper/selection.py
Docstring:   SelectionFunction(selection_data: Dict[str, jax.jaxlib._jax.Array], analysis_time: float = 1, total_generated: Optional[int] = None, redshift_model: Optional[gravpop.models.generic.abstract.AbstractPopulationModel] = None, total_detected: Optional[int] = None)

# Define Sampler

In [ ]:
samp = Sampler(
    priors=priors,
    latex_symbols={k:k for k in priors.keys()},
    likelihood=HL
);

# Run Sampler

In [13]:
samp.sample()

ValueError: Unit distribution got invalid log_factor parameter.

~ runs for 13 minutes...probably just doing the fits ahead of time. Also compiling stuff. Once that's done...takes like ~18 s to fail.

In [8]:
samp.sample()

ValueError: Unit distribution got invalid log_factor parameter.

In [7]:
samp.sample()

E0210 07:47:18.517711 1832346 slow_operation_alarm.cc:73] 
********************************
[Compiling module jit_while for GPU] Very slow compile? If you want to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
********************************
E0210 07:47:52.446933 1832188 slow_operation_alarm.cc:140] The operation took 2m33.929302163s

********************************
[Compiling module jit_while for GPU] Very slow compile? If you want to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
********************************


ValueError: Unit distribution got invalid log_factor parameter.

In [ ]:
samp.samples.to_csv("result_o4a_old_mass_model.csv")